# Lab 06 · Inspect saved results

Optional viewer. Run the processing commands in the terminal first. Open this notebook from the package directory. No processing logic lives here.

In [ ]:
from pathlib import Path
import json
import sqlite3
import rasterio
import matplotlib.pyplot as plt

ROOT = Path.cwd()
report = json.loads((ROOT / 'work/demo/report/report.json').read_text())
print(report['generated_at'], report['counts'])

The saved report describes an inspection at its recorded time. Regenerate it when you need current file evidence.

In [ ]:
for member in report['members']:
    result = member['result']
    metrics = result['receipt']['metrics'] if result else {}
    print(member['member_id'], member['state'], metrics.get('valid_fraction'), member['qc']['outcome'] if member['qc'] else 'unassessed')

In [ ]:
member = next(m for m in report['members'] if m['result'])
with rasterio.open(member['result']['paths']['index']) as source:
    image = source.read(1, masked=True)
plt.figure(figsize=(6, 4))
plt.imshow(image, cmap='BrBG', vmin=-1, vmax=1)
plt.title(member['member_id'])
plt.colorbar(label='MNDWI')
plt.xlabel('Column')
plt.ylabel('Row')
plt.show()

In [ ]:
database = (ROOT / 'work/demo/catalogue.sqlite').resolve()
with sqlite3.connect(database.as_uri() + '?mode=ro', uri=True) as connection:
    rows = connection.execute('SELECT outcome, COUNT(*) FROM latest_scientific_qc GROUP BY outcome').fetchall()
rows

Keep explanations and reusable SQL in your submission files. Your extension should run through the command line without this notebook.